# 04. Advanced Analysis — 고급 분석

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/dartlab/blob/master/notebooks/tutorials/04_advanced.ipynb)

K-IFRS 주석(Notes) 통합 접근, 유형자산 변동표, 관계기업 분석 등 심화 모듈을 다룬다. 모든 데이터는 property 또는 `get()`으로 접근한다.

**학습 내용:**
- K-IFRS 주석 통합 접근 (`c.notes`)
- 주석 세부항목 (23개 키워드)
- 유형자산 변동표
- 관계기업·공동기업 투자
- 이사회와 감사제도
- 거버넌스·리스크 분석
- 여러 종목 비교 분석

In [ ]:
!pip install -q dartlab

In [ ]:
from dartlab import Company

samsung = Company("005930")

## 1. K-IFRS 주석 — Notes 통합 접근

`c.notes`로 12개 K-IFRS 주석 항목에 통합 접근한다. 영문 속성과 한글 키 모두 지원한다.

In [ ]:
# 재고자산
samsung.notes.inventory

In [ ]:
# 매출채권
samsung.notes.receivables

In [ ]:
# 한글 키로도 접근 가능
samsung.notes["차입금"]

In [ ]:
# 전체 키 목록
print("영문 키:", samsung.notes.keys())
print("한글 키:", samsung.notes.keys_kr())

## 2. 주석 세부항목 — 23개 키워드

12개 주요 키워드 외에도 23개 키워드를 지원한다. `get("notesDetail", keyword=...)`으로 직접 호출한다.

지원 키워드: `재고자산`, `주당이익`, `충당부채`, `차입금`, `매출채권`, `리스`, `투자부동산`, `무형자산`, `법인세`, `특수관계자`, `약정사항`, `금융자산`, `공정가치`, `이익잉여금`, `금융부채`, `기타포괄손익`, `사채`, `종업원급여`, `퇴직급여`, `확정급여`, `재무위험`, `우발부채`, `담보`

### 연도별 상세 테이블

In [ ]:
inventory = samsung.get("notesDetail", keyword="재고자산")

print(f"키워드: {inventory.keyword}")
print(f"분석 연도: {inventory.nYears}")

In [ ]:
# 요약 테이블
inventory.tableDf

In [ ]:
# 연도별 상세 테이블
for year, periods in inventory.tables.items():
    for p in periods:
        print(f"[{year}] {p.period} (패턴: {p.pattern})")
        for item in p.items[:3]:
            print(f"  {item.name}: {item.values}")
        print()

## 3. 유형자산 변동표

토지, 건물, 기계장치 등 카테고리별 취득·처분·감가상각을 추적한다.

In [ ]:
# Notes 경유
samsung.notes.tangibleAsset

In [ ]:
# get()으로 전체 Result
ta = samsung.get("tangibleAsset")
print(f"신뢰도: {ta.reliability}")
if ta.warnings:
    print(f"경고: {ta.warnings}")

In [ ]:
# 카테고리별 기초/기말 시계열
ta.movementDf

## 4. 관계기업·공동기업

관계기업의 지분율, 장부가, 변동내역을 분석한다.

In [ ]:
# Notes 경유 — 변동 시계열
samsung.notes.affiliates

In [ ]:
# get()으로 개별 프로필
aff = samsung.get("affiliates")
if aff:
    for year, profiles in aff.profiles.items():
        for p in profiles[:3]:
            print(f"[{year}] {p.name}: {p.ratio}%, 장부가 {p.bookValue}")

## 5. 이사회와 감사제도

거버넌스 관련 데이터를 property로 접근한다.

In [ ]:
# 이사회 시계열
samsung.boardOfDirectors

In [ ]:
# 감사제도
samsung.auditSystem

In [ ]:
# 내부통제
samsung.internalControl

In [ ]:
# get()으로 이사회 위원회 구성
board = samsung.get("boardOfDirectors")
board.committeeDf

In [ ]:
# get()으로 감사활동 내역
audit_sys = samsung.get("auditSystem")
audit_sys.activityDf

## 6. 여러 종목 비교 분석

property 접근으로 간결하게 비교 분석할 수 있다.

### 재무제표 비교

In [ ]:
import polars as pl

codes = ["005930", "000660", "035420"]
rows = []

for code in codes:
    c = Company(code)
    bs = c.BS
    if bs is not None:
        cols = [col for col in bs.columns if col != "account"]
        last_year = cols[-1]
        row = bs.filter(pl.col("account") == "자산총계")
        if row.height > 0:
            rows.append({
                "기업": c.corpName,
                "자산총계": row[last_year][0],
            })

pl.DataFrame(rows)

### 배당 비교

In [ ]:
dividends = []
for code in codes:
    c = Company(code)
    div = c.dividend
    if div is not None:
        last = div.row(-1, named=True)
        dividends.append({
            "기업": c.corpName,
            "DPS": last.get("dps"),
            "배당수익률": last.get("dividendYield"),
            "배당성향": last.get("payoutRatio")
        })

pl.DataFrame(dividends)

## 7. 거버넌스·리스크 종합

관계자거래, 우발부채, 제재현황, 위험관리 등을 property로 조회한다.

In [ ]:
# 관계자거래
samsung.relatedPartyTx

In [ ]:
# 우발부채
samsung.contingentLiability

In [ ]:
# 제재 현황
samsung.sanction

In [ ]:
# 위험관리
samsung.riskDerivative

In [ ]:
# get()으로 관계자거래 — 매출입 거래
rpt = samsung.get("relatedPartyTx")
rpt.revenueTxDf

In [ ]:
# 관계자 채무보증
rpt.guaranteeDf

In [ ]:
# get()으로 우발부채 — 소송 현황
cl = samsung.get("contingentLiability")
cl.lawsuitDf

## 8. Result 객체 접근

property는 대표 DataFrame 하나를 반환한다. 모듈의 전체 데이터가 필요하면 `get()`을 사용한다.

In [ ]:
# property → 대표 DataFrame
samsung.audit

In [ ]:
# get() → 감사보수 (property로는 안 보이는 추가 데이터)
result = samsung.get("audit")
result.feeDf

## 9. 전체 일괄 조회

`all()`을 호출하면 40개 모듈을 순회하면서 전체 데이터를 dict로 반환한다.

In [ ]:
d = samsung.all()
list(d.keys())

In [ ]:
d["BS"]

In [ ]:
d["notes"]["inventory"]